## **Title:**
#### **Unsupervised Socioeconomic Profiling: Structuring Global Economic Indicators via Multi-Model Clustering and Dimensionality Reduction**

In [132]:
import pandas as pd

df_Data = pd.read_excel(
    r"D:\Projects\Data\Countries_Socioeconom_Profiles_Data\World_Development_Indicators.xlsx",
    sheet_name=0
)

In [133]:
import numpy as np

df_Data = df_Data.replace("..", np.nan)

In [134]:
df_Data

,Country Name,Country Code,Series Name,Series Code,2010 [YR2010],2011 [YR2011],2012 [YR2012],2013 [YR2013],2014 [YR2014],2015 [YR2015],2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024],2025 [YR2025]
0,Afghanistan,AFG,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,14.362441,0.426355,12.752287,5.600745,2.724543,1.451315,2.260314,2.647003,1.189228,3.911603,-2.351101,-20.738839,-6.240172,2.266944,1.873193,NaN
1,Afghanistan,AFG,GDP per capita (constant 2015 US$),NY.GDP.PCAP.KD,542.87103,525.426983,568.929021,580.603833,575.146246,565.56973,563.872337,562.769574,553.125152,557.861533,527.834554,408.625855,377.665627,378.066303,374.376696,NaN
2,Afghanistan,AFG,"Agriculture, forestry, and fishing, value adde...",NV.AGR.TOTL.ZS,26.210069,23.743664,24.390874,22.810663,22.137041,20.634323,25.740314,26.420199,22.042897,25.773971,29.975583,33.597619,33.701432,34.743247,34.2931,NaN
3,Afghanistan,AFG,"Industry (including construction), value added...",NV.IND.TOTL.ZS,21.151421,22.740252,21.157807,20.444605,21.229663,22.124042,10.466808,10.051874,13.387247,14.058112,12.9526,14.273657,16.050368,13.449823,14.520351,NaN
4,Afghanistan,AFG,"Services, value added (% of GDP)",NV.SRV.TOTL.ZS,48.879377,49.69682,50.579398,52.682743,52.990865,53.235293,59.023216,58.32905,59.477423,55.472807,52.572109,47.160422,45.049909,46.37437,45.161472,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9800,Zimbabwe,ZWE,Imports of goods and services (% of GDP),NE.IMP.GNFS.ZS,53.483016,54.665462,48.999912,36.668717,33.741765,37.589025,31.27546,30.370142,28.386265,18.483962,16.828444,19.289881,24.844552,23.174185,23.447532,NaN
9801,Zimbabwe,ZWE,Research and development expenditure (% of GDP),GB.XPD.RSDV.GD.ZS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9802,Zimbabwe,ZWE,"Fertility rate, total (births per woman)",SP.DYN.TFRT.IN,4.04,4.126,4.134,4.111,4.011,3.911,3.828,3.768,3.744,3.748,3.754,3.765,3.767,3.724,3.674,NaN
9803,Zimbabwe,ZWE,Current health expenditure (% of GDP),SH.XPD.CHEX.GD.ZS,10.475753,8.081707,6.918387,7.110222,8.133513,7.452041,7.258456,6.174489,4.67328,3.232682,2.954401,2.671119,3.395195,2.926016,NaN,NaN


In [135]:
df_Data.isna().sum()

Country Name        0
Country Code        0
Series Name         0
Series Code         0
2010 [YR2010]    1991
2011 [YR2011]    1983
2012 [YR2012]    1994
2013 [YR2013]    1967
2014 [YR2014]    2001
2015 [YR2015]    2017
2016 [YR2016]    2062
2017 [YR2017]    2215
2018 [YR2018]    2273
2019 [YR2019]    2352
2020 [YR2020]    2411
2021 [YR2021]    2473
2022 [YR2022]    2915
2023 [YR2023]    3086
2024 [YR2024]    4061
2025 [YR2025]    6730
dtype: int64

In [136]:
df_Data = df_Data.drop(columns=['2022 [YR2022]', '2023 [YR2023]', '2024 [YR2024]', '2025 [YR2025]'])

In [137]:
# Year columns
year_cols = [f"{year} [YR{year}]" for year in range(2010, 2022)]

df_Data = df_Data.melt(
    id_vars=["Country Name", "Country Code", "Series Name", "Series Code"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Value"
)

# Extract only the year
df_Data["Year"] = df_Data["Year"].str.extract(r"(\d{4})").astype(int)

# Make values numeric
df_Data["Value"] = pd.to_numeric(df_Data["Value"], errors="coerce")

In [138]:
df_Data["Value"] = pd.to_numeric(df_Data["Value"], errors="coerce")

In [139]:
df_wide = df_Data.pivot(
    index=["Country Name", "Country Code", "Year"],
    columns="Series Name",
    values="Value"
).reset_index()

In [140]:
import re

df_wide.columns = (
    df_wide.columns
    .str.lower()
    .str.strip()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.replace(r"\s+", "_", regex=True)
)

In [141]:
rename_dict = {
    "Country Name": "country_name",
    "Country Code": "country_code",
    "Year": "year",
    
    "Access to electricity (% of population)": "access_to_electricity",
    "Agriculture, forestry, and fishing, value added (% of GDP)": "agriculture_value_added",
    "Broad money (% of GDP)": "broad_money",
    "Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita)": "co2_emissions_per_capita",
    "Carbon intensity of GDP (kg CO2e per constant 2015 US$ of GDP)": "carbon_intensity_gdp",
    "Central government debt, total (% of GDP)": "government_debt",
    "Current health expenditure (% of GDP)": "health_expenditure",
    "Domestic credit to private sector (% of GDP)": "domestic_credit_private_sector",
    "Energy use (kg of oil equivalent per capita)": "energy_use_per_capita",
    "Exports of goods and services (% of GDP)": "exports",
    "Fertility rate, total (births per woman)": "fertility_rate",
    "Foreign direct investment, net inflows (% of GDP)": "fdi_inflows",
    "Forest area (% of land area)": "forest_area",
    "Fossil fuel energy consumption (% of total)": "fossil_fuel_consumption",
    "GDP growth (annual %)": "gdp_growth",
    "GDP per capita (constant 2015 US$)": "gdp_per_capita",
    "Gross capital formation (% of GDP)": "gross_capital_formation",
    "Imports of goods and services (% of GDP)": "imports",
    "Individuals using the Internet (% of population)": "internet_usage",
    "Industry (including construction), value added (% of GDP)": "industry_value_added",
    "Inflation, consumer prices (annual %)": "inflation",
    "Life expectancy at birth, total (years)": "life_expectancy",
    "Literacy rate, adult total (% of people ages 15 and above)": "literacy_rate",
    "Military expenditure (% of general government expenditure)": "military_expenditure",
    "Multidimensional poverty headcount ratio (World Bank) (% of population)": "multidimensional_poverty",
    "Population growth (annual %)": "population_growth",
    "Renewable electricity output (% of total electricity output)": "renewable_electricity",
    "Renewable energy consumption (% of total final energy consumption)": "renewable_energy",
    "Research and development expenditure (% of GDP)": "rd_expenditure",
    "School enrollment, primary (% gross)": "primary_school_enrollment",
    "Services, value added (% of GDP)": "services_value_added",
    "Trade (% of GDP)": "trade",
    "Unemployment, total (% of total labor force) (modeled ILO estimate)": "unemployment",
    "Urban population (% of total population)": "urban_population"
}

df_wide = df_wide.rename(columns=rename_dict)

In [142]:
df_wide = df_wide.sort_values(by=['year', 'country_name'])

In [143]:
df_wide.isna().sum().sort_values()

Series Name
country_name                                                                    0
country_code                                                                    0
year                                                                            0
fertility_rate_total_births_per_woman                                          12
life_expectancy_at_birth_total_years                                           12
population_growth_annual_                                                      12
urban_population_of_total_population                                           12
access_to_electricity_of_population                                            36
forest_area_of_land_area                                                       55
gdp_growth_annual_                                                             96
gdp_per_capita_constant_2015_us                                               104
renewable_energy_consumption_of_total_final_energy_consumption                123
rene

In [144]:
import pycountry

# Retain only actual countries with valid ISO 3166-1 alpha-3 codes
valid_iso_codes = {country.alpha_3 for country in pycountry.countries}
df_countries = df_wide[df_wide['country_code'].isin(valid_iso_codes)].copy()

In [145]:
feature_cols = df_countries.columns.difference(
    ["country_name", "country_code", "year"]
)

df_country_nan = (
    df_countries.groupby(["country_name", "country_code"])[feature_cols]
    .apply(lambda x: x.isna().sum().sum())
    .rename("nan_count")
    .reset_index()
)

# Total possible values per country
df_country_nan["total_values"] = (
    df_countries.groupby(["country_name", "country_code"])
    .size()
    .values * len(feature_cols)
)

# Percentage of missing values
df_country_nan["nan_percentage"] = (
    df_country_nan["nan_count"] /
    df_country_nan["total_values"]
)

df_country_nan = df_country_nan.sort_values(
    "nan_percentage",
    ascending=False
)

df_country_nan

,country_name,country_code,nan_count,total_values,nan_percentage
181,St. Martin (French part),MAF,354,444,0.797297
74,Gibraltar,GIB,315,444,0.709459
27,British Virgin Islands,VGB,311,444,0.700450
93,Isle of Man,IMN,306,444,0.689189
102,"Korea, Dem. People's Rep.",PRK,299,444,0.673423
...,...,...,...,...,...
177,Spain,ESP,29,444,0.065315
26,Brazil,BRA,28,444,0.063063
152,Peru,PER,15,444,0.033784
58,El Salvador,SLV,13,444,0.029279


In [146]:
df_country_nan.describe()

,nan_count,total_values,nan_percentage
count,215.000000,215.0,215.000000
mean,100.483721,444.0,0.226315
std,68.862917,0.0,0.155097
min,12.000000,444.0,0.027027
25%,51.000000,444.0,0.114865
50%,77.000000,444.0,0.173423
75%,127.000000,444.0,0.286036
max,354.000000,444.0,0.797297


In [147]:
nan_per_delete = df_country_nan['nan_percentage'].quantile(q= 0.95)

In [148]:
country_to_delete = df_country_nan[df_country_nan['nan_percentage'] >= nan_per_delete]

In [149]:
country_to_delete_list = country_to_delete['country_name'].tolist()

In [150]:
for country in country_to_delete_list:

    df_countries = df_countries[df_countries['country_name'] != country]

In [151]:
nan_summary = (
    pd.DataFrame({
        "nan_count": df_countries.isna().sum(),
        "nan_percentage": df_countries.isna().mean()
    })
    .reset_index(names="Series Name")
    .sort_values("nan_percentage", ascending=False)
)

nan_summary

,Series Name,nan_count,nan_percentage
28,literacy_rate_adult_total_of_people_ages_15_an...,1898,0.775327
8,central_government_debt_total_of_gdp,1829,0.747141
12,expenditure_on_primary_education_of_government...,1749,0.714461
13,expenditure_on_secondary_education_of_governme...,1740,0.710784
14,expenditure_on_tertiary_education_of_governmen...,1693,0.691585
30,multidimensional_poverty_headcount_ratio_world...,1632,0.666667
34,research_and_development_expenditure_of_gdp,1337,0.546160
19,fossil_fuel_energy_consumption_of_total,731,0.298611
11,energy_use_kg_of_oil_equivalent_per_capita,686,0.280229
29,military_expenditure_of_general_government_exp...,678,0.276961


In [152]:
nan_summary.describe()

,nan_count,nan_percentage
count,40.000000,40.000000
mean,458.850000,0.187439
std,616.207156,0.251719
min,0.000000,0.000000
25%,30.500000,0.012459
50%,171.500000,0.070057
75%,630.750000,0.257659
max,1898.000000,0.775327


In [153]:
null_per_max = 0.5

Series_Name_delete = nan_summary[nan_summary['nan_percentage'] >= null_per_max]
Series_Name_delete_list = Series_Name_delete['Series Name'].tolist()

In [154]:
df_countries = df_countries.drop(columns=Series_Name_delete_list)    